# MartiniSurf Colab: DNA + auto_martini M2
### No-code workflow (forms only)

## 1) Install MartiniSurf

In [ ]:
#@title Install MartiniSurf (supports private repo via token)
REPO_URL = "https://github.com/jjimenezgar/MartiniSurf.git" #@param {type:"string"}
PRIVATE_REPO = True #@param {type:"boolean"}
INSTALL_VIEWER = True #@param {type:"boolean"}

import subprocess
from pathlib import Path

repo_dir = Path('/content/MartiniSurf')
subprocess.run('rm -rf /content/MartiniSurf', shell=True, check=False)

if PRIVATE_REPO:
    import getpass
    token = getpass.getpass('Paste your GitHub token (with repo read access): ')
    if not token:
        raise RuntimeError('Token is required for private repository installation.')
    if REPO_URL.startswith('https://'):
        clone_url = REPO_URL.replace('https://', f'https://{token}@', 1)
    else:
        raise RuntimeError('REPO_URL must be https://...')
else:
    clone_url = REPO_URL

subprocess.run(f"git clone '{clone_url}' /content/MartiniSurf", shell=True, check=True)
subprocess.run('pip -q install --upgrade pip', shell=True, check=True)
subprocess.run('pip -q install -e /content/MartiniSurf', shell=True, check=True)
if INSTALL_VIEWER:
    subprocess.run('pip -q install py3Dmol', shell=True, check=True)

print('MartiniSurf installed successfully.')


## 2) Optional linker generation (SMILES or uploaded file)

In [ ]:
#@title Optional linker generation (auto_martini M2)
RUN_LINKER_GENERATION = False #@param {type:"boolean"}
INPUT_MODE = "SMILES" #@param ["SMILES", "Upload file"]
SMILES = "CCO" #@param {type:"string"}
MOLECULE_NAME = "LNK" #@param {type:"string"}

from pathlib import Path
import subprocess
import shlex

if RUN_LINKER_GENERATION:
    out_dir = Path('/content/generated_linker_m2')
    out_dir.mkdir(parents=True, exist_ok=True)

    repo = Path('/content/auto_martini')
    subprocess.run('rm -rf /content/auto_martini', shell=True, check=False)
    subprocess.run('git clone https://github.com/tbereau/auto_martini /content/auto_martini', shell=True, check=True)

    req = repo / 'requirements.txt'
    if req.exists():
        subprocess.run(f"pip -q install -r {shlex.quote(str(req))}", shell=True, check=False)

    scripts = ['auto_martini.py', 'automartini.py', 'AutoMartini.py']
    script_paths = [str((repo / s).resolve()) for s in scripts if (repo / s).exists()]
    if not script_paths:
        for cand in repo.rglob('*.py'):
            n = cand.name.lower()
            if 'martini' in n and 'auto' in n:
                script_paths.append(str(cand.resolve()))

    if not script_paths:
        raise RuntimeError('Could not find auto_martini executable script in cloned repository.')

    if INPUT_MODE == 'Upload file':
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('No file uploaded.')
        infile = '/content/' + next(iter(uploaded.keys()))
        input_specs = [('--mol', infile), ('--input', infile), ('-i', infile)]
        smiles_specs = []
    else:
        smiles_specs = [('--smi', SMILES), ('--smiles', SMILES)]
        input_specs = []

    name_flags = ['--molname', '--name']
    out_flags = ['--output', '--out']

    attempts = []
    if smiles_specs:
        for script in script_paths:
            for smi_flag, smi_val in smiles_specs:
                for name_flag in name_flags:
                    for out_flag in out_flags:
                        attempts.append(['python', script, smi_flag, smi_val, name_flag, MOLECULE_NAME, out_flag, str(out_dir)])
    else:
        for script in script_paths:
            for in_flag, in_val in input_specs:
                for name_flag in name_flags:
                    for out_flag in out_flags:
                        attempts.append(['python', script, in_flag, in_val, name_flag, MOLECULE_NAME, out_flag, str(out_dir)])

    ok = False
    last_err = ''
    for cmd in attempts:
        print('Trying:', ' '.join(shlex.quote(x) for x in cmd))
        r = subprocess.run(cmd, text=True, capture_output=True)
        if r.returncode == 0:
            ok = True
            break
        last_err = (r.stderr or r.stdout or '').strip()

    if not ok:
        print("Last command output:\n" + last_err[:3000])
        raise RuntimeError(
            'auto_martini command failed. The upstream CLI may have changed; '
            'check /content/auto_martini/README.md for exact flags.'
        )

    print('Done. Generated files:')
    subprocess.run(f"ls -lh '{out_dir}'", shell=True, check=False)
    print('Tip: upload/copy generated linker .gro and matching .itp before enabling linker mode below.')
else:
    print('Skipped.')


## 3) Build system

In [ ]:
#@title Build DNA system
PDB_INPUT_MODE = "PDB ID or filename" #@param ["PDB ID or filename", "Upload PDB file"]
PDB_INPUT = "4C64.pdb" #@param {type:"string"}
DNATYPE = "ds-stiff" #@param ["ds-stiff", "ds-flex", "ss"]
OUTDIR = "Simulation_Files" #@param {type:"string"}

USE_EXISTING_SURFACE = True #@param {type:"boolean"}
SURFACE_SOURCE = "Upload surface.gro now" #@param ["Upload surface.gro now", "Use existing surface.gro in session"]
SURFACE_FILE = "surface.gro" #@param {type:"string"}
LX = 5.0 #@param {type:"number"}
LY = 5.0 #@param {type:"number"}
SURFACE_BEAD = "C1" #@param {type:"string"}

USE_LINKER = False #@param {type:"boolean"}
LINKER_SOURCE = "Upload linker files now" #@param ["Upload linker files now", "Use existing linker.gro/.itp in session"]
LINKER_GRO = "linker.gro" #@param {type:"string"}
INVERT_LINKER = False #@param {type:"boolean"}

ANCHOR_1 = "1 1" #@param {type:"string"}
ANCHOR_2 = "2 24" #@param {type:"string"}
DIST = 10.0 #@param {type:"number"}

from pathlib import Path
import subprocess, shlex

if PDB_INPUT_MODE == 'Upload PDB file':
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No PDB uploaded')
    pdb_input = '/content/' + next(iter(uploaded.keys()))
else:
    pdb_input = PDB_INPUT

if USE_EXISTING_SURFACE:
    if SURFACE_SOURCE == 'Upload surface.gro now':
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('No surface file uploaded')
        surface_path = '/content/' + next(iter(uploaded.keys()))
    else:
        surface_path = '/content/' + SURFACE_FILE
        if not Path(surface_path).exists():
            raise FileNotFoundError(f'Missing file: {surface_path}')

cmd = ['martinisurf', '--dna', '--pdb', pdb_input, '--dnatype', DNATYPE, '--outdir', OUTDIR]

if USE_EXISTING_SURFACE:
    cmd += ['--surface', surface_path]
else:
    cmd += ['--lx', str(LX), '--ly', str(LY), '--surface-bead', SURFACE_BEAD]

if USE_LINKER:
    if LINKER_SOURCE == 'Upload linker files now':
        from google.colab import files
        print('Upload linker.gro and linker.itp (matching basename)')
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError('No linker files uploaded')
        names = list(uploaded.keys())
        gros = [n for n in names if n.endswith('.gro')]
        if not gros:
            raise RuntimeError('Please upload a linker .gro file')
        linker_path = '/content/' + gros[0]
    else:
        linker_path = '/content/' + LINKER_GRO
        if not Path(linker_path).exists():
            raise FileNotFoundError(f'Missing linker file: {linker_path}')

    cmd += ['--linker', linker_path]
    if INVERT_LINKER:
        cmd += ['--invert-linker']
    cmd += ['--linker-group'] + ANCHOR_1.split()
else:
    cmd += ['--anchor'] + ANCHOR_1.split()
    if ANCHOR_2.strip():
        cmd += ['--anchor'] + ANCHOR_2.split()
    cmd += ['--dist', str(DIST)]

print("Running:\n" + ' '.join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True)
print('Build completed')


## 4) Download

In [ ]:
#@title Download results as ZIP
OUTPUT_FOLDER = "Simulation_Files" #@param {type:"string"}
ZIP_NAME = "Simulation_Files" #@param {type:"string"}

import shutil
from pathlib import Path
from google.colab import files

folder = Path('/content') / OUTPUT_FOLDER
if not folder.exists():
    raise FileNotFoundError(f'Folder not found: {folder}')

shutil.make_archive(ZIP_NAME, 'zip', str(folder))
files.download(ZIP_NAME + '.zip')


## 5) Visualize final structure

In [ ]:
#@title Viewer (generated structure)
SYSTEM_GRO_FILE = "Simulation_Files/2_system/immobilized_system.gro" #@param {type:"string"}
STYLE = "Spheres" #@param ["Spheres", "Sticks"]

import py3Dmol
from pathlib import Path

path = Path('/content') / SYSTEM_GRO_FILE
if not path.exists():
    raise FileNotFoundError(f'File not found: {path}')

model = path.read_text()
view = py3Dmol.view(width='100%', height=800)
view.addModel(model, 'gro')
if STYLE == 'Sticks':
    view.setStyle({'stick': {}})
else:
    view.setStyle({'sphere': {'radius': 1.4}})
view.zoomTo()
view.show()


## Acknowledgements and Licenses
This notebook uses third-party tools and libraries with their own licenses:

- **martinize2** (Martini Force Field Initiative, Apache-2.0) when relevant
- **auto_martini (M2)** (tbereau/auto_martini, see repository license)
- **martinize-dna.py** workflow (legacy Martini DNA tooling; see upstream licensing)
- **MartiniSurf** (this repository)
- Python ecosystem packages used here, including (depending on step): `numpy`, `scipy`, `MDAnalysis`, `mdtraj`, `py3Dmol`

Please cite and comply with each upstream project license when publishing results.
